In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
import numpy as np
import time
from IPython.display import clear_output

class ADXL362:
    G_TO_MS2 = 9.81
    
    # Register addresses
    RESET     = 0x1F
    FILTER    = 0x2C
    POWER_CTL = 0x2D
    TEMP_L    = 0x14
    TEMP_H    = 0x15
    
    def __init__(self):
        print("Initializing ADXL362...")
        self.base = BaseOverlay('base.bit')
        self.lib = MicroblazeLibrary(self.base.PMODA, ['spi'])
        self.device = self.lib.spi_open(3, 2, 1, 0)  # SCLK, MISO, MOSI, CS
        
        if not self.device:
            raise RuntimeError("Failed to initialize SPI device")
            
        self.device.configure(0, 0)  # Mode 0
        time.sleep(0.1)
        
    def read_reg(self, addr):
        cmd = np.array([0x0B, addr, 0], dtype=np.uint8)
        data = np.zeros(3, dtype=np.uint8)
        self.device.transfer(cmd, data, 3)
        return int(data[2])
        
    def write_reg(self, addr, value):
        value = value & 0xFF
        cmd = np.array([0x0A, addr, value], dtype=np.uint8)
        data = np.zeros(3, dtype=np.uint8)
        self.device.transfer(cmd, data, 3)
        time.sleep(0.001)  # Short delay after writes
        
    def init_sensor(self):
        # Soft reset first
        self.write_reg(self.RESET, 0x52)
        time.sleep(0.1)  # Wait for reset
        
        # Read IDs
        devid = self.read_reg(0x00)
        mstid = self.read_reg(0x01)
        partid = self.read_reg(0x02)
        
        print("Device ID: 0x{:02X} (Expected: 0xAD)".format(devid))
        print("MEMS ID: 0x{:02X} (Expected: 0x1D)".format(mstid))
        print("Part ID: 0x{:02X} (Expected: 0xF2)".format(partid))
        
        if devid != 0xAD or mstid != 0x1D or partid != 0xF2:
            return False
            
        print("\nSetting up sensor...")
        
        # Put in standby for configuration
        self.write_reg(self.POWER_CTL, 0x00)
        time.sleep(0.1)
        
        # Configure filter control:
        # Bits 7:6 = 00 (±2g range)
        # Bit 5 = 0 (reserved)
        # Bit 4 = 0 (half bandwidth)
        # Bit 3 = 0 (external sampling disabled)
        # Bits 2:0 = 011 (ODR = 100Hz)
        self.write_reg(self.FILTER, 0x13)
        
        # Configure power control:
        # Bits 7:4 = 0000 (reserved and noise settings off)
        # Bit 3 = 0 (no wake-up)
        # Bit 2 = 0 (no autosleep)
        # Bits 1:0 = 10 (measurement mode)
        self.write_reg(self.POWER_CTL, 0x02)
        time.sleep(0.1)
        
        # Verify settings
        filter_val = self.read_reg(self.FILTER)
        power_val = self.read_reg(self.POWER_CTL)
        print("Filter Control: 0x{:02X} (Expected: 0x13)".format(filter_val))
        print("Power Control: 0x{:02X} (Expected: 0x02)".format(power_val))
        
        print("Setup complete!")
        return True
        
    def read_xyz(self):
        # Read all axis data at once to ensure consistency
        xl = self.read_reg(0x0E)
        xh = self.read_reg(0x0F)
        yl = self.read_reg(0x10)
        yh = self.read_reg(0x11)
        zl = self.read_reg(0x12)
        zh = self.read_reg(0x13)
        
        # Combine bytes into 12-bit values
        x = ((xh & 0x0F) << 8) | xl
        y = ((yh & 0x0F) << 8) | yl
        z = ((zh & 0x0F) << 8) | zl
        
        # Convert to signed values
        x = np.int16(x if x < 0x800 else -(0x1000 - x))
        y = np.int16(y if y < 0x800 else -(0x1000 - y))
        z = np.int16(z if z < 0x800 else -(0x1000 - z))
        
        # Convert to m/s² (first convert to g, then multiply by 9.81)
        scale = 1000.0  # For ±2g range
        return (x/scale) * self.G_TO_MS2, (y/scale) * self.G_TO_MS2, (z/scale) * self.G_TO_MS2
        
    def close(self):
        if hasattr(self, 'device'):
            self.lib.spi_close(self.device)

def display_readings(x, y, z):
    clear_output(wait=True)
    
    output = []
    output.append("ADXL362 Real-Time Accelerometer Readings")
    output.append("----------------------------------------")
    output.append("X-axis: {:7.3f} m/s²".format(x))
    output.append("Y-axis: {:7.3f} m/s²".format(y))
    output.append("Z-axis: {:7.3f} m/s²".format(z))
    output.append("\nPress Ctrl+C to exit")
    
    def make_bar(value, scale=8/9.81):  # Scale for visualization
        center = 40
        length = int(abs(value) * scale)
        length = min(length, center - 1)  # Limit bar length
        if value > 0:
            return " " * center + "#" * length + " {:7.3f}".format(value)
        else:
            return " " * (center - length) + "#" * length + " " * (center - (center - length)) + " {:7.3f}".format(value)
    
    output.append("\nGraphical View (each # = ~1.226 m/s²):")
    output.append("X: " + make_bar(x))
    output.append("Y: " + make_bar(y))
    output.append("Z: " + make_bar(z))
    
    print('\n'.join(output))

try:
    print("Starting ADXL362 test...")
    accel = ADXL362()
    
    if accel.init_sensor():
        print("\nStarting real-time display...")
        time.sleep(1)
        
        while True:
            x, y, z = accel.read_xyz()
            display_readings(x, y, z)
            time.sleep(0.05)  # 20Hz update rate
            
    else:
        print("Failed to initialize sensor!")
        
except KeyboardInterrupt:
    print("\nExiting...")
except Exception as e:
    print("Error: {}".format(str(e)))
finally:
    if 'accel' in locals():
        accel.close()

ADXL362 Real-Time Accelerometer Readings
----------------------------------------
X-axis:   0.059 m/s²
Y-axis:   0.186 m/s²
Z-axis:  11.644 m/s²

Press Ctrl+C to exit

Graphical View (each # = ~1.226 m/s²):
X:                                            0.059
Y:                                            0.186
Z:                                         #########  11.644
